In [1]:
import numpy as np
import pandas as pd
from nltk.classify import NaiveBayesClassifier
from nltk.corpus import subjectivity
from nltk.sentiment import SentimentAnalyzer
from nltk.sentiment.util import *

import numpy as np 
import pandas as pd 
#pd.set_option('display.max_colwidth', -1)
from time import time
import re
import string
import os

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import classification_report

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
#from sklearn.externals import joblib
import joblib as jb

#import gensim

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

import warnings
warnings.filterwarnings('ignore')

np.random.seed(37)

from nltk.sentiment.vader import SentimentIntensityAnalyzer
import unicodedata

In [2]:
# Reading the saved data pickle file
df_articles = pd.read_pickle('/Users/Farid Srouji/Documents/Research/VIXInterplaySentiment/master/financialmodelingprep/data/pkl/pickled_filtered_lead_para_sp500_clean.pkl', compression=None)

In [3]:
df  = pd.DataFrame()

# Adding new columns to the data frame
df[['compound']] = ''
df[['neg']] = ''
df[['neu']] = ''
df[['pos']] = ''
df[['sentiment']] = ''

In [4]:
debug=False
sid = SentimentIntensityAnalyzer()
countNeg=0
countNeu=0
countPos=0
countNegWord=0
countNeuWord=0
countPosWord=0

for date, row in df_articles.T.iteritems():
    print("The date is:"+str(date))
    try:
        sentence = unicodedata.normalize('NFKD', df_articles.loc[date, 'articles'])#.encode('ascii','ignore').decode('ascii')
        #sentence = unicodedata.normalize('NFKD', df_vix.loc[date, 'articles']).encode('ascii','ignore')
        # add my own sentence Options sentiment.
        #print("#1")
        #print (df_vix.loc[date, 'articles'])
        ss = sid.polarity_scores(sentence)
        #print(sentence)
        #print("#2")
        df.at[date, 'compound'] = ss['compound']
        df.at[date, 'neg'] = ss['neg']
        df.at[date, 'neu'] = ss['neu']
        df.at[date, 'pos'] = ss['pos']
        if ((ss['pos'] + ss['neg']) != 0):
            number = 100*(ss['pos'] / (ss['pos'] + ss['neg']))
            if (number != ''):
                df.at[date, 'sentiment'] = 100*(ss['pos'] / (ss['pos'] + ss['neg']))
                df.at[date, 'sentiment_complementary'] = 100 - 100*(ss['pos'] / (ss['pos'] + ss['neg']))

        countNeg+=ss['neg']
        countNeu+=ss['neu']
        countPos+=ss['pos']
        
        # print("============sentiment status============")
        # print("compound="+str(df.at[date, 'compound']))
        # print("neg="+str(df.at[date, 'neg']))
        # print("neu="+str(df.at[date, 'neu']))
        # print("pos="+str(df.at[date, 'pos']))
        # print("sentiment="+str(df.at[date, 'sentiment']))

        if (debug==True):
            words = sentence.split()
            countWordsNeg+=ss['neg']
            countWordsNeu+=ss['neu']
            countWordsPos+=ss['pos']
            for word in words:
                ww = sid.polarity_scores(word)
                countNeg+=ss['neg']
                countNeu+=ss['neu']
                countPos+=ss['pos']
            print("----------start sentenct-------")
            for word in words:
                try:
                    #print("word="+word)
                    ww = sid.polarity_scores(word)
                    if (ww['neg']==1):
                        countNegWord+=1
                    elif (ww['neu']==1):
                        countNeuWord+=1
                    elif (ww['pos']==1):
                        countPosWord+=1
                    else :
                        print("Error ww="+ww)
                except:
                    print("Exception for word="+word)

            print("----------end sentenct-------")

        #print("passed")
    except TypeError:
        #print("sentence is")
        #print(sentence)
        #print(df_vix.loc[date, 'articles'])
        print("ERROR")
        df.at[date, 'compound'] = 0
        df.at[date, 'neg'] = 0
        df.at[date, 'neu'] = 0
        df.at[date, 'pos'] = 0
        # df.at[date, 'sentiment'] = 0
        # df.at[date, 'sentiment_complementary'] = 0

The date is:2019-09-25
The date is:2019-09-26
The date is:2019-09-27
The date is:2019-09-30


In [ ]:

print("Number of Negative words="+str(countNeg))
print("Number of Positive words="+str(countPos))
print("Number of Neutral words="+str(countNeu))

df.to_pickle("/Users/Farid Srouji/Documents/Research/VIXInterplaySentiment/master/financialmodelingprep/data/pkl/sentiment.pkl")